# 06 · Model Comparison and Low-x Extrapolation

This notebook trains all supervised models and compares them on:

1. **Test-set performance metrics** (MSE, MAE, R²)
2. **Predictions vs data** at fixed Q² values
3. **Low-x extrapolation** — the central physics goal

The key question: which method gives the most physically reasonable
extrapolation into the uncharted low-x region?

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams["figure.dpi"] = 120

from data_loader import load_lepton_dis, add_log_features, split_data, get_Xy, low_x_grid
from visualization import plot_coverage, plot_F2_vs_x, comparison_figure

DATA_DIR = "/Users/dikgarg/Desktop/Research/Neutrinos/postdoc/2025/ML/Data/Exp_data/LeptonDIS"

df_raw = load_lepton_dis(DATA_DIR)
df     = add_log_features(df_raw)
df_train, df_test = split_data(df, test_size=0.2, seed=42)

X_train, y_train = get_Xy(df_train)
X_test,  y_test  = get_Xy(df_test)

print(f"Training points: {len(X_train)}  |  Test points: {len(X_test)}")


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import tensorflow as tf
import warnings
warnings.filterwarnings("ignore")
tf.random.set_seed(42)


## Train All Models

In [ ]:
# ── Scaler ────────────────────────────────────────────────────────────────
scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# ── Ridge ──────────────────────────────────────────────────────────────────
ridge = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=5)
ridge.fit(X_train_s, y_train)

# ── Polynomial deg 3 ───────────────────────────────────────────────────────
poly3 = Pipeline([("sc", StandardScaler()),
                  ("pf", PolynomialFeatures(3, include_bias=False)),
                  ("lr", LinearRegression())])
poly3.fit(X_train, y_train)

# ── GPR Matern ─────────────────────────────────────────────────────────────
kernel = (C(1.0, (1e-3, 1e3))
          * Matern(length_scale=[1., 1.], length_scale_bounds=(1e-2, 1e2), nu=1.5)
          + WhiteKernel(noise_level=0.01, noise_level_bounds=(1e-4, 1.0)))
gpr = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=3,
                                normalize_y=True, random_state=42)
gpr.fit(X_train_s, y_train)

# ── Random Forest ──────────────────────────────────────────────────────────
rf = RandomForestRegressor(n_estimators=400, min_samples_leaf=2,
                            random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# ── XGBoost ────────────────────────────────────────────────────────────────
xgb_m = xgb.XGBRegressor(n_estimators=600, learning_rate=0.05, max_depth=5,
                           subsample=0.8, random_state=42, verbosity=0)
xgb_m.fit(X_train, y_train, verbose=False)

# ── Bottleneck NN ──────────────────────────────────────────────────────────
inp = tf.keras.Input(shape=(2,))
x   = tf.keras.layers.Dense(64, activation="tanh")(inp)
x   = tf.keras.layers.Dense(32, activation="tanh")(x)
x   = tf.keras.layers.Dense(2,  activation="linear", name="bottleneck")(x)
x   = tf.keras.layers.Dense(32, activation="tanh")(x)
x   = tf.keras.layers.Dense(64, activation="tanh")(x)
out = tf.keras.layers.Dense(1,  activation="linear")(x)
bnn = tf.keras.Model(inp, out)
bnn.compile(optimizer=tf.keras.optimizers.Adam(3e-3), loss="mse")
bnn.fit(X_train_s.astype("float32"), y_train.astype("float32"),
        validation_data=(X_test_s.astype("float32"), y_test.astype("float32")),
        epochs=800, batch_size=64, verbose=0,
        callbacks=[tf.keras.callbacks.EarlyStopping(patience=50,
                                                     restore_best_weights=True)])
print("All models trained.")


## Test-Set Metrics

In [ ]:
def metrics(name, y_true, y_pred):
    return {"Model": name,
            "MSE":  round(mean_squared_error(y_true, y_pred), 5),
            "MAE":  round(mean_absolute_error(y_true, y_pred), 5),
            "R2":   round(r2_score(y_true, y_pred), 4)}

rows = [
    metrics("Ridge",         y_test, ridge.predict(X_test_s)),
    metrics("Poly-3",        y_test, poly3.predict(X_test)),
    metrics("GPR-Matern",    y_test, gpr.predict(X_test_s)),
    metrics("RandomForest",  y_test, rf.predict(X_test)),
    metrics("XGBoost",       y_test, xgb_m.predict(X_test)),
    metrics("Bottleneck-NN", y_test, bnn.predict(X_test_s.astype("float32"),
                                                  verbose=0).ravel()),
]
comp_df = pd.DataFrame(rows).sort_values("MSE").reset_index(drop=True)
print(comp_df.to_string(index=False))
comp_df.to_csv("../results/metrics_comparison.csv", index=False)


## Low-x Extrapolation — All Methods

In [ ]:
Q2_plot = [1.0, 5.0, 15.0, 30.0]
grids   = low_x_grid(x_min=1e-6, x_max=0.8, n_points=400, Q2_values=Q2_plot)

MODEL_SPEC = [
    ("Ridge",        "#a65628", lambda X: ridge.predict(scaler.transform(X))),
    ("Poly-3",       "#ff7f00", lambda X: poly3.predict(X)),
    ("GPR-Matern",   "#984ea3", lambda X: gpr.predict(scaler.transform(X))),
    ("RandomForest", "#e41a1c", lambda X: rf.predict(X)),
    ("XGBoost",      "#377eb8", lambda X: xgb_m.predict(X)),
    ("Bottleneck-NN","#4daf4a", lambda X: bnn.predict(
                                    scaler.transform(X).astype("float32"),
                                    verbose=0).ravel()),
]

predictions = []
for label, color, pred_fn in MODEL_SPEC:
    entry = {"label": label, "color": color, "x_arr": None, "Q2_preds": {}}
    for Q2v, (x_arr, X_feat) in grids.items():
        entry["x_arr"]         = x_arr
        entry["Q2_preds"][Q2v] = pred_fn(X_feat)
    predictions.append(entry)

# Also add GPR uncertainty
gpr_entry = next(p for p in predictions if p["label"] == "GPR-Matern")
gpr_entry["Q2_lo"] = {}
gpr_entry["Q2_hi"] = {}
for Q2v, (x_arr, X_feat) in grids.items():
    mu, sig = gpr.predict(scaler.transform(X_feat), return_std=True)
    gpr_entry["Q2_lo"][Q2v] = mu - 2 * sig
    gpr_entry["Q2_hi"][Q2v] = mu + 2 * sig

fig, _ = comparison_figure(Q2_plot, df, predictions, figsize=(18, 14))
fig.suptitle("Low-x extrapolation — all supervised methods", fontsize=14, y=1.01)
plt.savefig("../results/figures/06_all_models.png", dpi=150, bbox_inches="tight")
plt.show()


## Interpretation

| Method | Low-x behaviour |
|---|---|
| **Ridge / Poly-3** | Smooth power-law extrapolation — may over- or under-shoot |
| **GPR** | Principled uncertainty quantification; reverts to prior in data-free region |
| **Random Forest / XGBoost** | Plateau at the minimum training x — cannot extrapolate |
| **Bottleneck NN** | Smooth extrapolation from learned internal representation |

**Key take-away**: GPR is the most honest tool for extrapolation because it explicitly
signals when its prediction is uncertain. Tree methods should not be used for
out-of-sample extrapolation in kinematic variables.
